# 02 · Exploración de datos

Este notebook revisa la base integrada antes del modelado: tamaño, faltantes, balance de la etiqueta y distribuciones básicas. Ejecuta primero `python pipeline.py` o `06_pipeline_completo.ipynb` si todavía no existe el Parquet.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pipeline.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontró pipeline.py en la carpeta actual ni en sus padres.")
DATA_PATH = ROOT / "data" / "processed" / "dataset_enaho_escale_2025.parquet"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Falta {DATA_PATH}; ejecuta primero pipeline.py.")
data = pd.read_parquet(DATA_PATH)
print(f"{len(data):,} filas × {data.shape[1]} columnas")

## ¿Qué revisamos y por qué?

- Los faltantes indican qué variables podrían necesitar imputación dentro del entrenamiento.
- El balance muestra cuán frecuente es la etiqueta en la cohorte etiquetada.
- Estas tablas son descriptivas: aquí todavía no se ajusta ningún modelo.

In [ ]:
missing = data.isna().agg(["sum", "mean"]).T.rename(columns={"sum": "faltantes", "mean": "proporcion_faltante"})
display(missing.sort_values("proporcion_faltante", ascending=False).head(20))
display(data["exclusion_digital"].value_counts(dropna=False).rename_axis("clase").to_frame("casos"))
display(data[["edad", "ingreso_per_capita", "gasto_per_capita", "escale_ratio_alumnos_docente"]].describe().T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
data["edad"].dropna().hist(ax=axes[0], bins=12)
axes[0].set(title="Edades de estudiantes", xlabel="Edad", ylabel="Casos")
data["exclusion_digital"].dropna().astype(int).value_counts().sort_index().plot.bar(ax=axes[1])
axes[1].set(title="Etiqueta operacional", xlabel="0 = usa Internet; 1 = no usa", ylabel="Casos")
plt.tight_layout()